# TODO:
1. Save processed sdata file to original zarr file

# Packages

In [1]:
# import cellcharter

import cell2location

import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import spatialdata as spd
import anndata
import scanpy as sc

/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/janzules/miniforge3/envs/spatial_cpu_py311/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
import pandas as pd
import os
import mygene # Grabbing ENSEMBLE ID

In [4]:
# Run only on the HPC CUDA node
# import torch
# print(f"Cuda available? {torch.cuda.is_available()}")
# print(f"Cuda device: {torch.cuda.get_device_name(0)}")

# Functions

In [4]:
def del_DS_Store(root):
    """
    Recursively delete all .DS_Store files under 'root'.
    and read zarr.
    """
    root = Path(root)

    for p in root.rglob(".DS_Store"):
        p.unlink()


    

# Data

## Laptop

In [5]:
# Locations
proj_folder  = Path("/Users/janzules/Roselab/Spatial/CAR_T/")
data_folder  = proj_folder / "data"
zarr_loc     = data_folder / "zarrFiles/CART_centroid"
sc_ref_loc   = data_folder / "sc-reference/sc_reference_cell2location.h5ad"

# Output location
results_folder = proj_folder / "Results/cell2location"
ref_run_name   = results_folder / "reference_signatures"
run_name       = results_folder / "cell2location_map"

In [7]:
# for c in zarr_loc.rglob(".DS_Store"):
#     print(c)
#     print(f"Is this a file? {c.is_file()}")

In [8]:
# os.mkdir(ref_run_name)
# os.mkdir(run_name)

In [6]:
# Loading
# del_DS_Store(zarr_loc) # Removing random .DS_Store added to file tree because Macbook

zarr_spatial = spd.read_zarr(zarr_loc)
sdata        = zarr_spatial.tables['segmentation_counts']
del zarr_spatial # Don't need for now

adata_ref    = sc.read_h5ad(sc_ref_loc)

/var/folders/np/h5smkry919g1mwf1wtltfqtm0000gp/T/ipykernel_1987/3877133078.py:4: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  zarr_spatial = spd.read_zarr(zarr_loc)


# Pre processing

In [7]:
sdata.obs.head()

,sample,cell_id,region,TMA,mouse,tissue,condition,tumor_loc,replicate_num
F07839_cellid_000000001-1,F07839,F07839_cellid_000000001-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000002-1,F07839,F07839_cellid_000000002-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000004-1,F07839,F07839_cellid_000000004-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000005-1,F07839,F07839_cellid_000000005-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1
F07839_cellid_000000006-1,F07839,F07839_cellid_000000006-1,CyPSCA_1_1_cell_boundaries,F07839,CyPSCA_1_1,CyPSCA_1_1,CyPSCA,1,1


## Compatible Naming Schemes

In [8]:
# Saving as integer
sdata.obs['tumor_loc'] = sdata.obs['tumor_loc'].astype(int)
sdata.obs['replicate_num'] = sdata.obs['replicate_num'].astype(int)

In [9]:
# Creating the column that will match the sc reference
sdata.obs['treatment'] = sdata.obs['condition']

tumor_locations = [1, 2]

for tumor in tumor_locations:

    if tumor == 1:
        tumor_num = 1
        suffix = "_Tu1"
    elif tumor == 2:
        tumor_num = 2
        suffix = "_Tu2"
    else:
        ValueError("Something went wrong")
        
    tum_1_mask = sdata.obs["tumor_loc"] == tumor_num
    
    sdata.obs.loc[tum_1_mask, 'treatment'] = (
        sdata.obs.loc[tum_1_mask, 'treatment']
        .str.replace(r"T72", "TAG72")
        + suffix
    )
# sdata.obs['treatment']


In [10]:
sdata.obs['treatment'].unique()

array(['CyPSCA_Tu1', 'CyPSCA_Tu2', 'CyTAG72_Tu1', 'CyTAG72_Tu2',
       'NoTx_Tu1', 'NoTx_Tu2', 'RTCyPSCA_Tu1', 'RTCyPSCA_Tu2',
       'RTCyTAG72_Tu2', 'RTCyTAG72_Tu1'], dtype=object)

### SC - Names 

#### Checking for dissimilarity

In [16]:
sdata_trmts = set(sdata.obs['treatment'].astype(str))
adata_trmts = set(adata_ref.obs['treatment'].astype(str))

In [17]:
for name in sorted(sdata_trmts):
    print(name)

print(f"Length of names: {len(sdata_trmts)}")

CyPSCA_Tu1
CyPSCA_Tu2
CyTAG72_Tu1
CyTAG72_Tu2
NoTx_Tu1
NoTx_Tu2
RTCyPSCA_Tu1
RTCyPSCA_Tu2
RTCyTAG72_Tu1
RTCyTAG72_Tu2
Length of names: 10


In [18]:
for name in sorted(adata_trmts):
    print(name)

print(f"Length of names: {len(adata_trmts)}")

CyPSCA_Tu1
CyPSCA_Tu2
CyTAG72_Tu1
CyTAG72_Tu2
NoTx_Tu1
NoTx_Tu2
RTCyPSCA_Tu1
RTCyPSCA_Tu2
RTCyTAG72_Tu1
RTCyTAG72_Tu2
Length of names: 10


In [19]:
onlyIn_sdata = sdata_trmts - adata_trmts
onlyIn_sdata

set()

In [20]:
onlyIn_adata = adata_trmts - sdata_trmts
onlyIn_adata

set()

In [21]:
onlyIn_sdata == onlyIn_sdata

True

#### Spot check for name

In [22]:
treats = [
    "CyPSCA_Tu1",
    "CyPSCA_Tu2",
    "CyTAG72_Tu1",
    "CyTAG72_Tu2",
    "NoTx_Tu1",
    "NoTx_Tu2",
    "RTCyPSCA_Tu1",
    "RTCyPSCA_Tu2",
    "RTCyTAG72_Tu1",
    "RTCyTAG72_Tu2",
]

In [23]:
for i in range(0,10, 1):
    print("")
    print("-----------------------------------------------")
    print(f"Checking treatment: {treats[i]}")
    treats_mask = sdata.obs['treatment'] == treats[i]
    subset_check = sdata[treats_mask].copy()
    
    print(subset_check.obs.loc[:,['treatment', 'condition', 'tumor_loc']].head())


-----------------------------------------------
Checking treatment: CyPSCA_Tu1
                            treatment condition  tumor_loc
F07839_cellid_000000001-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000002-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000004-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000005-1  CyPSCA_Tu1    CyPSCA          1
F07839_cellid_000000006-1  CyPSCA_Tu1    CyPSCA          1

-----------------------------------------------
Checking treatment: CyPSCA_Tu2
                            treatment condition  tumor_loc
F08542_cellid_000047107-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047110-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047111-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047112-1  CyPSCA_Tu2    CyPSCA          2
F08542_cellid_000047113-1  CyPSCA_Tu2    CyPSCA          2

-----------------------------------------------
Checking treatment: CyTAG72_Tu1
                             treatment condition  t

## ENSEMBL

In [11]:
sdata.var['SYMBOL'] = sdata.var_names

In [12]:
symbols = pd.Index(sdata.var_names).astype(str)

print("n genes:", len(symbols))
print("example:", symbols[:10].tolist())

n genes: 19059
example: ['Xkr4', 'Rp1', 'Sox17', 'Lypla1', 'Tcea1', 'Rgs20', 'Atp6v1h', 'Oprk1', 'Npbwr1', 'Rb1cc1']


In [ ]:
mg = mygene.MyGeneInfo()

res = mg.querymany(
    symbols.tolist(),
    scopes="symbol",
    fields="ensembl.gene,symbol",
    species="mouse",
    verbose = True
)

In [70]:
# Converting list of dicts into 
map_df = pd.DataFrame(res)

# Testing

In [58]:
map_df.head()

,query,_id,_score,ensembl,symbol,notfound
0,Xkr4,497097,14.785570,{'gene': 'ENSMUSG00000051951'},Xkr4,NaN
1,Rp1,19888,15.440001,{'gene': 'ENSMUSG00000025900'},Rp1,NaN
2,Sox17,20671,15.149700,{'gene': 'ENSMUSG00000025902'},Sox17,NaN
3,Lypla1,18777,14.852126,{'gene': 'ENSMUSG00000025903'},Lypla1,NaN
4,Tcea1,21399,14.743513,{'gene': 'ENSMUSG00000033813'},Tcea1,NaN


In [84]:
na_df     = map_df.loc[map_df['ensembl'].isna()].sample(n=5, random_state=0)
not_na_df = map_df.loc[map_df['ensembl'].notna()].sample(n=5, random_state=0)

combined = pd.concat([na_df, not_na_df], axis=0)

combined

,query,_id,_score,ensembl,symbol,notfound
1840,Olfr1217,NaN,NaN,NaN,NaN,True
15545,Olfr288,NaN,NaN,NaN,NaN,True
1190,Il1f5,NaN,NaN,NaN,NaN,True
1739,Olfr1097,NaN,NaN,NaN,NaN,True
14562,Olfr732,NaN,NaN,NaN,NaN,True
15256,Oplah,75475,14.843104,{'gene': 'ENSMUSG00000022562'},Oplah,NaN
12599,Tns4,217169,15.284526,{'gene': 'ENSMUSG00000017607'},Tns4,NaN
13188,Arhgap5,11855,14.764845,{'gene': 'ENSMUSG00000035133'},Arhgap5,NaN
9419,Znrf1,170737,14.768924,{'gene': 'ENSMUSG00000033545'},Znrf1,NaN
758,Tor1aip2,240832,16.388847,{'gene': 'ENSMUSG00000050565'},Tor1aip2,NaN


In [97]:
def testing_NA_check(x):
    # if x is None:
    if pd.isna(x):
        return "Oi, me blank"
        # return np.nan
    if isinstance(x, dict):
        return x.get("gene", np.nan)
        
    return type(x)

In [98]:
combined['TEST'] = combined['ensembl'].apply(testing_NA_check)
combined

,query,_id,_score,ensembl,symbol,notfound,TEST
1840,Olfr1217,NaN,NaN,NaN,NaN,True,"Oi, me blank"
15545,Olfr288,NaN,NaN,NaN,NaN,True,"Oi, me blank"
1190,Il1f5,NaN,NaN,NaN,NaN,True,"Oi, me blank"
1739,Olfr1097,NaN,NaN,NaN,NaN,True,"Oi, me blank"
14562,Olfr732,NaN,NaN,NaN,NaN,True,"Oi, me blank"
15256,Oplah,75475,14.843104,{'gene': 'ENSMUSG00000022562'},Oplah,NaN,ENSMUSG00000022562
12599,Tns4,217169,15.284526,{'gene': 'ENSMUSG00000017607'},Tns4,NaN,ENSMUSG00000017607
13188,Arhgap5,11855,14.764845,{'gene': 'ENSMUSG00000035133'},Arhgap5,NaN,ENSMUSG00000035133
9419,Znrf1,170737,14.768924,{'gene': 'ENSMUSG00000033545'},Znrf1,NaN,ENSMUSG00000033545
758,Tor1aip2,240832,16.388847,{'gene': 'ENSMUSG00000050565'},Tor1aip2,NaN,ENSMUSG00000050565


In [75]:
NoMatch = map_df.loc[map_df['ensembl'].isna()].copy()
NoMatch.head()

,query,_id,_score,ensembl,symbol,notfound
80,Pih1d3,NaN,NaN,NaN,NaN,True
96,Arhgef4-1,NaN,NaN,NaN,NaN,True
223,Fam126b,NaN,NaN,NaN,NaN,True
255,Gpr1,NaN,NaN,NaN,NaN,True
296,March4,NaN,NaN,NaN,NaN,True


In [76]:
NoMatch['TEST'] = NoMatch['ensembl'].apply(testing_NA_check).copy()
NoMatch.head()

,query,_id,_score,ensembl,symbol,notfound,TEST
80,Pih1d3,NaN,NaN,NaN,NaN,True,NaN
96,Arhgef4-1,NaN,NaN,NaN,NaN,True,NaN
223,Fam126b,NaN,NaN,NaN,NaN,True,NaN
255,Gpr1,NaN,NaN,NaN,NaN,True,NaN
296,March4,NaN,NaN,NaN,NaN,True,NaN


In [ ]:
map_df[map_df['ensembl'] == np.nan]

In [62]:
isinstance(map_df['ensembl'][3], dict)

True

## Understanding ensemble column

In [73]:

map_df.loc[3:5,:]

,query,_id,_score,ensembl,symbol,notfound
3,Lypla1,18777,14.852126,{'gene': 'ENSMUSG00000025903'},Lypla1,NaN
4,Tcea1,21399,14.743513,{'gene': 'ENSMUSG00000033813'},Tcea1,NaN
5,Rgs20,58175,14.810723,{'gene': 'ENSMUSG00000002459'},Rgs20,NaN


In [60]:
map_df['ensembl']

0        {'gene': 'ENSMUSG00000051951'}
1        {'gene': 'ENSMUSG00000025900'}
2        {'gene': 'ENSMUSG00000025902'}
3        {'gene': 'ENSMUSG00000025903'}
4        {'gene': 'ENSMUSG00000033813'}
                      ...              
19087    {'gene': 'ENSMUSG00000064363'}
19088    {'gene': 'ENSMUSG00000064367'}
19089    {'gene': 'ENSMUSG00000064368'}
19090    {'gene': 'ENSMUSG00000064370'}
19091                               NaN
Name: ensembl, Length: 19092, dtype: object

In [15]:
type(map_df['ensembl'][0])

dict

In [16]:
map_df['ensembl'][0]['gene']

'ENSMUSG00000051951'

In [17]:
map_df['ensembl'][0].get('gene')

'ENSMUSG00000051951'

In [17]:
map_df['ensembl'][0]

'ENSMUSG00000051951'